# AI 이미지 분석기 서버
FastAPI와 Ollama(gemma3:4b)를 사용하여 이미지를 분석하는 서버입니다.

In [ ]:
import os
import uvicorn
import ollama
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv()

app = FastAPI(title="AI Image Analyzer")

# CORS 설정: 모든 출처 허용
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/analyze")
async def analyzeImage(imageFile: UploadFile = File(...)):
    """
    업로드된 이미지를 Ollama gemma3:4b 모델을 사용하여 분석합니다.
    
    Args:
        imageFile (UploadFile): 분석할 이미지 파일
        
    Returns:
        dict: 분석 결과 또는 에러 메시지
    """
    try:
        # 이미지 데이터 읽기
        imageBytes = await imageFile.read()
        
        # Ollama 모델 호출
        targetModel = os.getenv("OLLAMA_MODEL", "gemma3:4b")
        
        analysisResponse = ollama.chat(
            model=targetModel,
            messages=[{
                'role': 'user',
                'content': 'Describe this image in detail.',
                'images': [imageBytes]
            }]
        )
        
        return {
            "status": "success",
            "analysisResult": analysisResponse['message']['content']
        }
        
    except Exception as errorInstance:
        # 예외 처리 및 상세 에러 로깅
        print(f"Error occurred: {str(errorInstance)}")
        raise HTTPException(status_code=500, detail="이미지 분석 중 오류가 발생했습니다.")

if __name__ == "__main__":
    # 서버 실행 설정
    serverPort = int(os.getenv("PORT", 8000))
    uvicorn.run(app, host="0.0.0.0", port=serverPort)